[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_25/Scania_APS_Colab_Notebook.ipynb)

# Erkennung von APS-Fehlern anhand von Betriebsdaten

Dieses Notebook zeigt ausgewählte Schritte einer Klassifikationsaufgabe: Daten prüfen, ein einfaches Vergleichsmodell und einen Random Forest erstellen sowie Fehlentscheidungen kostenbewusst bewerten. Es ist kein vollständiger CRISP-DM-Ablauf: Insbesondere werden das Geschäftsverständnis nur vereinfacht betrachtet und die Einführung in den Betrieb nicht umgesetzt.

**Quelle:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks). Das Archiv enthält 60.000 Trainings- und 16.000 Testfälle. Die Trainingsdaten enthalten 1.000 positive Fälle und 170 anonymisierte Betriebsmerkmale.

Die positive Klasse `pos` steht für Fehler an einer bestimmten Komponente des Druckluftsystems (APS). Die negative Klasse `neg` umfasst Fahrzeuge mit Fehlern an Komponenten außerhalb des APS; sie bedeutet also nicht unbedingt „fehlerfrei“.

**Wichtige Grenze:** Die Daten sind einzelne Beobachtungen ohne Zeitfolge. Das Modell unterscheidet Fehlerklassen, sagt aber weder den Zeitpunkt noch die verbleibende Zeit bis zu einem Fehler voraus. Ob es einen Ausfall frühzeitig erkennen kann, lässt sich damit nicht belegen. Es ist daher keine vollständige Lösung für vorausschauende Wartung (Predictive Maintenance). Die anonymisierten Merkmale erlauben außerdem nur eine eingeschränkte technische Interpretation.

Die UCI-Aufgabe setzt Kosten von 10 für einen Fehlalarm (FP) und 500 für einen nicht erkannten APS-Fehler (FN) fest. Diese Werte sind Annahmen für die Benchmark und müssen in einer realen Anwendung durch betriebliche Kosten ersetzt werden.

## 1. Daten laden und einen ersten Überblick gewinnen

Der Download enthält beide offiziellen Dateien. Für diese Übung verwenden wir nur den Trainingsdatensatz und teilen ihn anschließend stratifiziert in Trainings-, Validierungs- und Testdaten auf.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

In [ ]:
archive_url = "https://archive.ics.uci.edu/static/public/421/aps+failure+at+scania+trucks.zip"
with urlopen(archive_url) as response:
    archive_bytes = response.read()

with ZipFile(BytesIO(archive_bytes)) as archive:
    training_file = next(
        name for name in archive.namelist()
        if name.endswith("aps_failure_training_set.csv")
    )
    with archive.open(training_file) as csv_file:
        # Die ersten 20 Zeilen sind ein Lizenztext, die Spaltennamen stehen in Zeile 21
        df = pd.read_csv(csv_file, skiprows=20, na_values=["na", "NA"])

df.shape

In [ ]:
display(df.head())
display(df["class"].value_counts().rename("Anzahl"))
missing_rate = df.drop(columns="class").isna().mean().sort_values(ascending=False)
display(missing_rate.head(10).rename("Fehlender Anteil"))

df["class"].value_counts().plot.bar(color=["steelblue", "tomato"])
plt.title("Zielklassen im Trainingsdatensatz")
plt.ylabel("Anzahl der Fälle")
plt.xticks(rotation=0)
plt.show()

Ein Modell, das immer die Mehrheitsklasse `neg` vorhersagt, erreicht bereits eine Accuracy von etwa 98,3 %. Dieser Wert reicht als Erfolgsmessung nicht aus, denn das Modell erkennt damit keinen einzigen APS-Fehler der positiven Klasse. Durch stratifizierte Aufteilungen ist die seltene positive Klasse in allen Datenteilmengen vertreten.

In [ ]:
X = df.drop(columns="class")
y = df["class"].map({"neg": 0, "pos": 1})
if y.isna().any():
    raise ValueError("Unerwartete Werte in der Zielspalte 'class'.")

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid, test_size=0.25,
    stratify=y_train_valid, random_state=42
)
y_train.value_counts(), y_valid.value_counts(), y_test.value_counts()

## 2. Baseline und Modell

Zuerst bewerten wir einen Mehrheitsklassen-Klassifikator. Danach trainieren wir einen Random Forest. Der Median-Imputer wird in der Pipeline nur anhand der jeweiligen Trainingsdaten angepasst; dadurch gelangen keine Informationen aus Validierung oder Test in die Datenvorbereitung.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_valid)
print(classification_report(
    y_valid, baseline_predictions,
    target_names=["neg", "pos"], zero_division=0
))

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
)
model.fit(X_train, y_train)
valid_scores = model.predict_proba(X_valid)[:, 1]
print("PR-AUC (Validierung):", average_precision_score(y_valid, valid_scores))
print("ROC-AUC (Validierung):", roc_auc_score(y_valid, valid_scores))

## 3. Schwellenwert anhand der Fehlerkosten wählen

Ein niedrigerer Schwellenwert kann mehr positive Fälle erkennen, führt aber meist auch zu mehr Fehlalarmen. Wir wählen ihn anhand des Validierungsdatensatzes mit der Kostenfunktion der UCI-Aufgabe. Die Testdaten bleiben bis zur abschließenden Bewertung unberührt.

In [ ]:
# Auch kleine Schwellenwerte prüfen: Ein übersehener Fehler kostet deutlich mehr als ein Fehlalarm.
thresholds = np.arange(0.01, 1.00, 0.01)
validation_costs = []
for threshold in thresholds:
    predictions = (valid_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, predictions, labels=[0, 1]).ravel()
    validation_costs.append(10 * fp + 500 * fn)

best_threshold = thresholds[int(np.argmin(validation_costs))]
best_threshold, min(validation_costs)

## 4. Einmalige Bewertung auf dem Testdatensatz

Nach der Wahl des Schwellenwerts trainieren wir die Pipeline erneut auf Trainings- und Validierungsdaten. Die Testdaten dienen nur der abschließenden Schätzung.

In [ ]:
model.fit(X_train_valid, y_train_valid)
test_scores = model.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, test_predictions, labels=[0, 1]
).ravel()
print(classification_report(
    y_test, test_predictions,
    target_names=["neg", "pos"], zero_division=0
))
print("Gewählter Schwellenwert:", best_threshold)
print("PR-AUC (Test):", average_precision_score(y_test, test_scores))
print("ROC-AUC (Test):", roc_auc_score(y_test, test_scores))
print("UCI-Kostenmetrik (Test):", 10 * fp + 500 * fn)
print("Fehlalarme FP:", fp, "| nicht erkannte Fehler FN:", fn)

## 5. Einordnung und Reflexion

- Vergleicht Recall und Precision der positiven Klasse mit der Mehrheitsklassen-Baseline.
- Erklärt, warum sich die Anzahl der Fehlalarme und nicht erkannten APS-Fehler mit dem Schwellenwert verändert.
- Diskutiert, ob die UCI-Kostenannahmen für eine konkrete Wartungsorganisation plausibel wären.
- Benennt die Grenzen: anonymisierte Merkmale, fehlende Werte, keine zeitliche Abfolge und keine Aussage zur Ursache oder zum Zeitpunkt eines Fehlers.

Ein gutes Ergebnis auf diesem Benchmark belegt noch nicht, dass das Modell in einer realen Flotte sicher oder wirtschaftlich eingesetzt werden kann.